# TXPipe - Cluster shear calibrator

Test notebook for the `CLClusterShearCalibrator` stage (`txpipe/extensions/cluster_counts/cluster_shear_calibrator.py`).

The upstream stages of `tangential_shear_selection.ipynb` (source selection, cluster shear catalogs, cluster binning) are run once and their outputs are reused. After that, only the calibrator stage is run, and it is reloaded every time so edits to the stage are picked up without restarting the kernel.

In [ ]:
import os
import sys
import importlib
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import h5py
import ceci

In [ ]:
my_txpipe_dir = "/sps/lsst/users/ebarroso/TXPipe"
sys.path.append(my_txpipe_dir)
sys.path.append(f"{my_txpipe_dir}/txpipe/extensions")
import txpipe
import txpipe.extensions as extensions

## Files

All the outputs of this notebook go to `outputs_cluster_calibration`, so the other notebooks are not affected. Set `RERUN_UPSTREAM = True` to regenerate the upstream outputs.

In [ ]:
inputs_dir = f"{my_txpipe_dir}/data/example/inputs"
out_dir = f"{my_txpipe_dir}/data/example/outputs_cluster_calibration"
os.makedirs(out_dir, exist_ok=True)

shear_catalog = f"{inputs_dir}/metadetect_shear_catalog.hdf5"
cluster_catalog = f"{inputs_dir}/cluster_catalog.hdf5"
photoz_pdfs = f"{inputs_dir}/photoz_pdfs.hdf5"
fiducial_cosmology = f"{my_txpipe_dir}/data/fiducial_cosmology.yml"
shear_selection = f"{out_dir}/shear_selection.hdf5"
shear_bin_assignments = f"{out_dir}/shear_bin_assignments.hdf5"
shear_tomography_catalog = f"{out_dir}/shear_tomography_catalog.hdf5"

cluster_shear_catalogs = f"{out_dir}/cluster_shear_catalogs.hdf5"
cluster_catalog_tomography = f"{out_dir}/cluster_catalog_tomography.hdf5"
cluster_shear_response = f"{out_dir}/cluster_shear_response.hdf5"
cluster_shear_response_stack = f"{out_dir}/cluster_shear_response_stack.hdf5"

RERUN_UPSTREAM = False

---

## Upstream stages (run once)

Same stages and options as `tangential_shear_selection.ipynb`, with one configuration only: $\Delta\Sigma$ with zmode for the cut and the weights.

In [ ]:
if RERUN_UPSTREAM or not os.path.exists(shear_tomography_catalog):
    step1 = txpipe.TXMetaDetectSelection.make_stage(
        shear_catalog=shear_catalog,
        shear_selection=shear_selection,
        output_dir=out_dir,
    )
    step1.run()
    step1.finalize()

    # tomographic bins from the true redshifts
    step1b = txpipe.TXSimpleRedshiftSourceTomography.make_stage(
        shear_catalog=shear_catalog,
        shear_bin_assignments=shear_bin_assignments,
        output_dir=out_dir,
        source_zbin_edges=[0.5, 0.7, 0.9, 1.1, 2.0],
        chunk_rows = 100_000,
    )
    step1b.run()
    step1b.finalize()

    step1c = txpipe.TXComputeShearCalibrationMetadetect.make_stage(
        shear_catalog=shear_catalog,
        shear_bin_assignments=shear_bin_assignments,
        shear_selection=shear_selection,
        shear_tomography_catalog=shear_tomography_catalog,
        output_dir=out_dir,
        delta_gamma=0.02,
    )
    step1c.run()
    step1c.finalize()

In [ ]:
if RERUN_UPSTREAM or not os.path.exists(cluster_shear_catalogs):
    step2 = extensions.CLClusterShearCatalogs.make_stage(
        shear_catalog=shear_catalog,
        cluster_catalog=cluster_catalog,
        fiducial_cosmology=fiducial_cosmology,
        shear_tomography_catalog=shear_tomography_catalog,
        source_photoz_pdfs=photoz_pdfs,
        cluster_shear_catalogs=cluster_shear_catalogs,
        output_dir=out_dir,
        max_radius=10.0,
        use_radius=True,
        max_angle=40,
        delta_sigma=True,
        redshift_weight_criterion="zmode",
        redshift_cut_criterion="zmode",
        use_shape_noise=False,
    )
    step2.run()
    step2.finalize()

In [ ]:
if RERUN_UPSTREAM or not os.path.exists(cluster_catalog_tomography):
    step3 = extensions.CLClusterBinningRedshiftRichness.make_stage(
        cluster_catalog=cluster_catalog,
        zedge=[0.2, 0.4, 0.6, 0.8],
        richedge=[5, 10, 100],
        cluster_catalog_tomography=cluster_catalog_tomography,
    )
    step3.run()
    step3.finalize()

---

## Calibrator stage

Rerun from here after every change to `cluster_shear_calibrator.py`. `allow_reload` lets ceci register the reloaded class again under the same name.

In [ ]:
ceci.PipelineStage.allow_reload = True
from txpipe.extensions.cluster_counts import cluster_shear_calibrator
importlib.reload(cluster_shear_calibrator)
CLClusterShearCalibrator = cluster_shear_calibrator.CLClusterShearCalibrator
print(CLClusterShearCalibrator.config_options)

In [ ]:
calib_args = dict(
    cluster_catalog_tomography=cluster_catalog_tomography,
    cluster_shear_catalogs=cluster_shear_catalogs,
    fiducial_cosmology=fiducial_cosmology,
    shear_catalog=shear_catalog,
    shear_tomography_catalog=shear_tomography_catalog,
    shear_bin_assignments=shear_bin_assignments,
    shear_selection=shear_selection,
    output_dir=out_dir,
    r_min=1.3,
    r_max=3.0,
    nbins=3,
    bin_units="mpc",
)

# One response per radial bin of each stack
calib = CLClusterShearCalibrator.make_stage(
    cluster_shear_response=cluster_shear_response,
    radial_calibration=True,
    **calib_args,
)
calib.run()
calib.finalize()

# One response per stack, to compare
calib_stack = CLClusterShearCalibrator.make_stage(
    cluster_shear_response=cluster_shear_response_stack,
    radial_calibration=False,
    **calib_args,
)
calib_stack.run()
calib_stack.finalize()

## Clusters and sources of one richness-redshift bin

In [ ]:
with h5py.File(cluster_catalog_tomography, "r") as f:
    bin_names = list(f["cluster_bin"].keys())
print(bin_names)

bin_name = bin_names[0]
clusters, sources = calib.load_cluster_bin_sources(bin_name)
print(clusters.meta)
clusters

In [ ]:
print(sources.meta)
sources[:10]

Number of sources per cluster and radial bin. Sources with `radial_bin = -1` are outside `[r_min, r_max]`.

In [ ]:
nbins = len(calib.distance_bins) - 1
in_range = sources["radial_bin"] >= 0
print("Sources outside the radial range:", np.sum(~in_range))
print("Sources per radial bin:", np.bincount(sources["radial_bin"][in_range], minlength=nbins))

counts = np.zeros((len(clusters), nbins), dtype=int)
for i, cid in enumerate(clusters["cluster_id"]):
    sel = in_range & (sources["cluster_id"] == cid)
    counts[i] = np.bincount(sources["radial_bin"][sel], minlength=nbins)
for cid, c in zip(clusters["cluster_id"], counts):
    print(cid, c)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for b in range(nbins):
    sel = sources["radial_bin"] == b
    ax[0].hist(sources["distance"][sel], bins=50, range=(0, sources["distance"].max()), histtype="step", label=f"bin {b}")
ax[0].hist(sources["distance"][~in_range], bins=50, range=(0, sources["distance"].max()), color="gray", alpha=0.3, label="outside")
for edge in calib.distance_bins:
    ax[0].axvline(edge, color="k", lw=0.5)
ax[0].set_xlabel(f"Distance [{sources.meta['bin_units']}]")
ax[0].set_ylabel("Sources")
ax[0].legend()

# Weighted mean of all the pairs of the bin, a quick look only (not the CLMM stacking)
# weight, tangential and cross components of the same pairs, from cluster_shear_catalogs
# (sources is ordered by cluster, as load_cluster_bin_sources reads them)
with h5py.File(cluster_shear_catalogs, "r") as f:
    profile_type = f["index"].attrs["profile_type"]
    row = {int(c): i for i, c in enumerate(f["catalog/cluster_id"][:])}
    starts = f["catalog/cluster_sample_start"][:]
    counts = f["catalog/cluster_sample_count"][:]
    slices = [slice(starts[row[int(c)]], starts[row[int(c)]] + counts[row[int(c)]]) for c in clusters["cluster_id"]]
    pair_cols = {col: np.concatenate([f["index"][col][sl] for sl in slices]) for col in ["weight", "tangential_comp", "cross_comp"]}

r_mid = np.sqrt(calib.distance_bins[1:] * calib.distance_bins[:-1])
w = pair_cols["weight"]
tan = [np.sum((w * pair_cols["tangential_comp"])[sources["radial_bin"] == b]) / np.sum(w[sources["radial_bin"] == b]) for b in range(nbins)]
cross = [np.sum((w * pair_cols["cross_comp"])[sources["radial_bin"] == b]) / np.sum(w[sources["radial_bin"] == b]) for b in range(nbins)]
ax[1].plot(r_mid, tan, "o-", label="tangential")
ax[1].plot(r_mid, cross, "s--", label="cross")
ax[1].set_xscale("log")
ax[1].set_xlabel(f"R [{sources.meta['bin_units']}]")
ax[1].set_ylabel(profile_type)
ax[1].legend()
plt.tight_layout()
plt.show()

---

## Next: calibration

`source_index` is the row of each source in the shear catalog. The metadetect catalog has the groups below.

In [ ]:
with h5py.File(shear_catalog, "r") as f:
    for group in f["shear"]:
        print(group, f["shear"][group]["g1"].shape)

---

## Response

The stage saves, for each stack, the tangential response `R_T` = (R11 + R22) / 2 of each radial bin (`radial_calibration=True`) or of the whole stack (`radial_calibration=False`), as in Zhou et al. 2026 (arXiv:2606.21141, Eqs. 22–23), plus the full R and mu. On this example catalog the true response is 1 for every object and there is no magnification, so there is no radial trend to find: R is dominated by selection noise, because `psf_T_mean` is drawn independently in each metadetect variant. This only checks that the stage runs.

In [ ]:
with h5py.File(shear_tomography_catalog, "r") as f:
    R_global = f["response/R_2d"][:]
print("global R_2d:", np.round(R_global.ravel(), 4), " R_T =", np.round(0.5 * np.trace(R_global), 4))

with h5py.File(cluster_shear_response, "r") as fr, h5py.File(cluster_shear_response_stack, "r") as fs:
    print(dict(fr["calibration"].attrs))
    edges = fr["calibration"].attrs["distance_bins"]
    units = fr["calibration"].attrs["bin_units"]
    RT_radial = {name: g["R_T"][:] for name, g in fr["calibration"].items()}
    RT_stack = {name: g["R_T"][()] for name, g in fs["calibration"].items()}
    n_radial = {name: g.attrs["n_sources_per_radial_bin"] for name, g in fr["calibration"].items()}
    n_stack = {name: g.attrs["n_sources"] for name, g in fs["calibration"].items()}

for name in RT_radial:
    print(f"\n{name}: stack R_T = {RT_stack[name]:.4f}, n = {n_stack[name]}")
    for k, (RT, n) in enumerate(zip(RT_radial[name], n_radial[name])):
        print(f"  radial bin {k} [{edges[k]:.2f}, {edges[k+1]:.2f}] {units}: n = {n}, R_T = {RT:.4f}")

# R_T per radial bin (points) and per stack (dashed), global in black
r_mid = np.sqrt(edges[1:] * edges[:-1])
fig, ax = plt.subplots(figsize=(6, 4))
for name in RT_radial:
    line, = ax.plot(r_mid, RT_radial[name], "o-", label=name)
    ax.axhline(RT_stack[name], color=line.get_color(), ls="--", lw=0.8)
ax.axhline(0.5 * np.trace(R_global), color="k", lw=1)
ax.set_xscale("log")
ax.set_xlabel(f"Distance [{units}]")
ax.set_ylabel("$R_T$")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()